# 모듈 01: 첫 연결 - Hello LangChain

환경 설정(모듈 00)이 완료되었습니다. 이제 **실제로 AI와 대화**를 나눠봅니다!

---

## 이 노트북에서 할 것

| 단계 | 내용 |
|------|------|
| 1 | Gemini 모델에 연결 (ChatGoogleGenerativeAI) |
| 2 | 문자열로 직접 AI 호출하기 |
| 3 | 메시지 타입 이해하기 (HumanMessage, AIMessage, SystemMessage) |
| 4 | 메시지 리스트로 AI 호출하기 |
| 5 | 응답 객체 탐색하기 |
| 6 | stream()으로 실시간 출력하기 |

**예상 소요 시간**: 약 30분

> **전제 조건**: 모듈 00이 완료되어 `.env` 파일에 `GOOGLE_API_KEY`가 설정되어 있어야 합니다.

## 학습 목표

이 모듈을 완료하면 다음 세 가지를 할 수 있습니다:

1. **ChatGoogleGenerativeAI로 Gemini 연결**: LangChain을 통해 Gemini 모델에 연결하고, 두 가지 방식(문자열, 메시지 리스트)으로 호출하는 방법을 압니다.

2. **메시지 타입 구분**: `HumanMessage`, `AIMessage`, `SystemMessage`의 역할 차이를 이해하고, 시스템 프롬프트로 AI의 역할을 설정하는 방법을 압니다.

3. **invoke()와 stream() 차이 체감**: 결과를 한 번에 받는 `invoke()`와 실시간으로 조금씩 받는 `stream()`의 차이를 실행으로 직접 확인합니다.

## 전체 흐름도

이 노트북의 전체 흐름을 한눈에 보면 이렇습니다:

```
┌─────────────────────────────────────────────────────┐
│                                                     │
│  [1] .env 로드 + ChatGoogleGenerativeAI 초기화      │
│       ↓                                             │
│  [2] 방법 1: 문자열로 invoke()                      │
│       → response.content로 결과 확인                │
│       ↓                                             │
│  [3] 메시지 타입 이해 (Human / AI / System)         │
│       ↓                                             │
│  [4] 방법 2: 메시지 리스트로 invoke()               │
│       → SystemMessage로 역할 설정                   │
│       ↓                                             │
│  [5] 응답 객체 탐색 (metadata, 토큰 사용량)         │
│       ↓                                             │
│  [6] stream()으로 실시간 출력                       │
│                                                     │
└─────────────────────────────────────────────────────┘
```

> **팁**: 셀을 순서대로 실행하세요! 위에서 아래로 하나씩 `Shift + Enter`를 눌러 실행하면 됩니다.

---

## 섹션 2: 모델 연결

### ChatGoogleGenerativeAI란?

`ChatGoogleGenerativeAI`는 LangChain이 Gemini 모델과 대화할 수 있도록 만든 **연결 클래스**입니다.

### LLM vs ChatModel - 전화 vs 문자 비유

LangChain에는 두 종류의 모델 인터페이스가 있습니다:

| 구분 | LLM | ChatModel |
|------|-----|-----------|
| 비유 | 전화 통화 | 문자 메시지 |
| 입력 | 단순 문자열 | 메시지 리스트 |
| 출력 | 문자열 | AIMessage 객체 |
| 특징 | 한 덩어리로 주고받음 | 역할(발신자)이 구분됨 |

**전화 통화(LLM)**: 말을 건네면 말로 돌려받음. 누가 말했는지 구분 없음.

**문자 메시지(ChatModel)**: 대화 기록이 남고, 누가 보낸 메시지인지 역할이 구분됨.

> `ChatGoogleGenerativeAI`는 ChatModel이지만, 편의상 문자열도 직접 받을 수 있습니다.
> 두 가지 방법 모두 이 노트북에서 실습합니다.

In [6]:
# .env 로드 + ChatGoogleGenerativeAI 초기화
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# 프로젝트 루트의 .env 파일 로드 (이 노트북은 01_hello/ 폴더에 있음)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(notebook_dir)
env_path = os.path.join(project_root, '.env')

loaded = load_dotenv(env_path)

# API 키 확인
api_key = os.getenv('GOOGLE_API_KEY')
if api_key and api_key != 'your_api_key_here':
    print(f'[OK] GOOGLE_API_KEY 로드 성공: {api_key[:10]}...')
else:
    print('[FAIL] API 키가 없습니다. 모듈 00을 먼저 완료하세요.')

# 모델 초기화
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.7,
)

print()
print('[OK] ChatGoogleGenerativeAI 초기화 완료!')
print(f'모델: {llm.model}')
print(f'타입: {type(llm).__name__}')

[OK] GOOGLE_API_KEY 로드 성공: AIzaSyAlrJ...

[OK] ChatGoogleGenerativeAI 초기화 완료!
모델: gemini-2.5-flash
타입: ChatGoogleGenerativeAI


### 주요 파라미터 설명

모델을 초기화할 때 두 가지 핵심 파라미터를 설정했습니다:

#### `model` - 어떤 모델을 쓸지

| 모델명 | 특징 |
|--------|------|
| `gemini-2.5-flash` | 빠르고 비용 효율적 (이 커리큘럼 기본 모델) |
| `gemini-2.5-pro` | 더 강력하지만 느리고 비쌈 |

#### `temperature` - 창의성 조절 (0.0 ~ 2.0)

**온도계 비유**: 온도가 높을수록 AI가 더 창의적(예측 불가)하게 답합니다.

| 값 | 특징 | 적합한 용도 |
|----|------|------------|
| 0.0 | 항상 동일한 답변 | 코드 생성, 사실 질문 |
| 0.7 | 적당히 창의적 | 일반 대화, 설명 (기본값) |
| 1.5+ | 매우 창의적, 예측 불가 | 창작, 브레인스토밍 |

---

## 섹션 3: 방법 1 - 문자열 입력

### 가장 단순한 호출 방법

LangChain의 모든 모델은 `.invoke()` 메서드로 호출합니다.

**가장 단순한 방법**: 문자열을 직접 넣는 것!

```python
response = llm.invoke("여기에 질문을 적으면 됩니다")
print(response.content)  # AI의 답변 텍스트
```

**자판기 비유**
- 동전을 넣으면 음료가 나오듯, 질문(문자열)을 넣으면 답변(AIMessage)이 나옵니다
- `response.content`는 음료 캔을 열어 내용물을 꺼내는 것

> **참고**: `invoke()`는 AI의 응답이 **완전히 완성될 때까지 기다렸다가** 한 번에 반환합니다.

In [7]:
# 방법 1: 문자열을 직접 invoke() 에 전달
print('=== 방법 1: 문자열 입력 ===')
print()

response = llm.invoke('안녕하세요! LangChain을 처음 배우고 있습니다. 한 문장으로 응원해주세요!')

print('AI 응답:')
print(response.content)

=== 방법 1: 문자열 입력 ===

AI 응답:
LangChain의 첫걸음을 내딛는 당신, 분명 멋진 AI 세상을 열어갈 거예요! 화이팅!


In [8]:
# AIMessage 응답 객체 구조 탐색
# response는 단순 문자열이 아니라 AIMessage 객체입니다
print('=== AIMessage 객체 구조 탐색 ===')
print()

print(f'타입:        {type(response).__name__}')
print(f'content:     {response.content[:50]}...')
print(f'id:          {response.id}')
print()
print('response_metadata 키 목록:')
for key in response.response_metadata:
    print(f'  - {key}')
print()
print('usage_metadata:')
print(f'  {response.usage_metadata}')

=== AIMessage 객체 구조 탐색 ===

타입:        AIMessage
content:     LangChain의 첫걸음을 내딛는 당신, 분명 멋진 AI 세상을 열어갈 거예요! 화이팅!...
id:          lc_run--019cb369-1d4a-7391-bfab-cfdd31216698-0

response_metadata 키 목록:
  - finish_reason
  - model_name
  - safety_ratings
  - model_provider

usage_metadata:
  {'input_tokens': 19, 'output_tokens': 816, 'total_tokens': 835, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 790}}


---

## 섹션 4: 메시지 타입

### 3가지 메시지 타입

채팅 앱을 떠올려보세요. 카카오톡에는 내 메시지, 상대방 메시지, 그리고 시스템 알림이 있습니다.
LangChain의 메시지 타입도 똑같은 구조입니다:

| 메시지 타입 | 역할 | 카카오톡 비유 |
|------------|------|---------------|
| `HumanMessage` | 사용자(나)의 메시지 | 내가 보낸 말풍선 (오른쪽) |
| `AIMessage` | AI의 응답 메시지 | 상대방 말풍선 (왼쪽) |
| `SystemMessage` | AI에게 주는 지시사항 | 채팅방 입장 시 규칙 안내 |

**SystemMessage의 특별한 역할**

SystemMessage는 AI에게 "너는 이런 역할이야"라고 알려주는 **무대 감독**과 같습니다.
대화 시작 전에 설정하며, 사용자에게는 보이지 않지만 AI의 모든 답변에 영향을 미칩니다.

```
SystemMessage: "너는 친절한 한국어 요리 전문가야"
HumanMessage:  "파스타 어떻게 만들어요?"
AIMessage:     "안녕하세요! 파스타 만드는 법을 알려드릴게요..."
```

In [9]:
# 메시지 객체 직접 만들어보고 구조 확인
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# 각 메시지 타입 생성
system_msg = SystemMessage(content='당신은 친절한 한국어 AI 튜터입니다.')
human_msg = HumanMessage(content='LangChain이 뭔가요?')
ai_msg = AIMessage(content='LangChain은 LLM 애플리케이션을 만드는 프레임워크입니다.')

print('=== 메시지 객체 구조 확인 ===')
print()

for msg in [system_msg, human_msg, ai_msg]:
    print(f'타입:    {type(msg).__name__}')
    print(f'역할:    {msg.type}')
    print(f'내용:    {msg.content}')
    print()

=== 메시지 객체 구조 확인 ===

타입:    SystemMessage
역할:    system
내용:    당신은 친절한 한국어 AI 튜터입니다.

타입:    HumanMessage
역할:    human
내용:    LangChain이 뭔가요?

타입:    AIMessage
역할:    ai
내용:    LangChain은 LLM 애플리케이션을 만드는 프레임워크입니다.



---

## 섹션 5: 방법 2 - 메시지 리스트 입력

### 왜 메시지 리스트를 쓸까요?

문자열 방법은 간단하지만, AI에게 **역할**을 부여하거나 **대화 맥락**을 전달하려면 메시지 리스트가 필요합니다.

**배우 오디션 비유**
- 문자열 방법: 배우에게 그냥 "대사를 말해봐"라고 하는 것
- 메시지 리스트: 배우에게 "너는 탐정 역할이야 (SystemMessage). 용의자에게 이렇게 물어봐 (HumanMessage)"라고 하는 것

### SystemMessage(시스템 프롬프트)의 역할

SystemMessage는 다음을 설정할 수 있습니다:
- **역할**: "당신은 전문 번역가입니다"
- **말투**: "항상 존댓말을 사용하세요"
- **제약**: "답변은 3문장 이내로 하세요"
- **형식**: "JSON 형식으로만 답하세요"

```python
messages = [
    SystemMessage(content="역할 설정"),  # AI 지시사항
    HumanMessage(content="사용자 질문"),  # 실제 질문
]
response = llm.invoke(messages)
```

In [10]:
# 방법 2: 메시지 리스트로 invoke()
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content='당신은 친절한 한국어 AI 튜터입니다. 초보자도 이해할 수 있게 쉽고 간단하게 설명하세요.'),
    HumanMessage(content='LangChain이 뭔가요? 한 문장으로 설명해주세요.'),
]

print('=== 방법 2: 메시지 리스트 입력 ===')
print()
print(f'입력 메시지 수: {len(messages)}개')
print(f'  1. {messages[0].type}: {messages[0].content[:30]}...')
print(f'  2. {messages[1].type}: {messages[1].content}')
print()

response = llm.invoke(messages)

print('AI 응답:')
print(response.content)

=== 방법 2: 메시지 리스트 입력 ===

입력 메시지 수: 2개
  1. system: 당신은 친절한 한국어 AI 튜터입니다. 초보자도 이해할...
  2. human: LangChain이 뭔가요? 한 문장으로 설명해주세요.

AI 응답:
LangChain은 AI 모델(LLM)이 다양한 정보나 도구들과 연결되어 더 똑똑한 프로그램을 쉽게 만들 수 있도록 도와주는 도구 모음입니다.


In [11]:
# SystemMessage를 바꿔서 같은 질문에 다른 응답 비교
question = HumanMessage(content='파이썬이란 무엇인가요?')

# 역할 A: 5살 아이에게 설명하는 선생님
role_a = SystemMessage(content='당신은 5살 아이에게 설명하는 선생님입니다. 매우 쉬운 단어만 사용하세요.')

# 역할 B: 엄격한 컴퓨터공학 교수
role_b = SystemMessage(content='당신은 엄격한 컴퓨터공학 교수입니다. 정확한 기술 용어와 함께 학술적으로 설명하세요.')

print('=== 같은 질문, 다른 SystemMessage ===')
print(f'질문: {question.content}')
print()

response_a = llm.invoke([role_a, question])
print('[역할 A - 5살 아이 선생님]')
print(response_a.content)
print()

response_b = llm.invoke([role_b, question])
print('[역할 B - 컴퓨터공학 교수]')
print(response_b.content)

=== 같은 질문, 다른 SystemMessage ===
질문: 파이썬이란 무엇인가요?

[역할 A - 5살 아이 선생님]
안녕! 우리 친구 파이썬에 대해 궁금하구나?

음, 파이썬은 말이야... 컴퓨터랑 이야기하는 아주 특별한 방법이야.

우리 친구 로봇이 있다고 상상해 볼까? 🤖

이 로봇은 우리가 하는 말을 못 알아들어. "로봇아, 그림 그려줘!" 하면 "잉? 무슨 말이지?" 할 거야.

그래서 우리는 로봇이 알아듣는 **특별한 말**로 이야기해야 해. 그 특별한 말이 바로 **'파이썬'**이야!

파이썬으로 로봇에게 "네모 그려줘!", "동그라미 그려줘!", "재미있는 게임 만들어줘!" 하고 말할 수 있어.

파이썬은 컴퓨터에게 "이렇게 해!", "저렇게 해!" 하고 알려주는 착한 선생님 같은 거야.

재미있겠지? 😊

[역할 B - 컴퓨터공학 교수]
학생, '파이썬(Python)'에 대해 질문하는군요. 컴퓨터공학적 관점에서 정확하게 설명하겠습니다.

---

### Python이란 무엇인가?

Python은 1980년대 후반 CWI(Centrum Wiskunde & Informatica)의 Guido van Rossum에 의해 설계 및 구현이 시작되어 1991년 최초 공개된 **고급(high-level), 인터프리티드(interpreted), 동적 타입(dynamically typed), 다중 패러다임(multi-paradigm) 프로그래밍 언어**입니다. 그 명칭은 영국의 코미디 그룹 'Monty Python'에서 유래했습니다.

Python은 명료하고 간결한 문법을 통해 높은 가독성(readability)과 생산성(productivity)을 제공하며, 광범위한 응용 분야에서 활용될 수 있도록 설계되었습니다.

#### 1. 주요 특성 (Key Characteristics)

*   **고급 언어 (High-Level Language):**
    하드웨어의 저수준(low-level) 세부 사항(예: 메모리 관리)으로부터 프로그래머를 추상화(abst

---

## 섹션 6: 응답 객체 탐색

### response 객체의 주요 속성

`llm.invoke()` 의 반환값은 단순 문자열이 아닌 **AIMessage 객체**입니다.
이 객체 안에는 텍스트 외에도 유용한 정보가 담겨 있습니다.

**택배 상자 비유**
- `response.content` = 상자 안의 물건 (실제 AI 답변 텍스트)
- `response.id` = 운송장 번호 (이 응답의 고유 ID)
- `response.response_metadata` = 배송 기록 (언제, 어떤 경로로 왔는지)
- `response.usage_metadata` = 포장 비용 청구서 (토큰 사용량)

### 토큰(Token)이란?

AI는 글자 수가 아닌 **토큰** 단위로 요금을 계산합니다.
- 영어: 단어 1개 ≈ 토큰 1개
- 한국어: 글자 1~2개 ≈ 토큰 1개
- `input_tokens`: 내가 보낸 질문의 토큰 수
- `output_tokens`: AI가 생성한 답변의 토큰 수

In [12]:
# response_metadata 출력 (모델명, finish_reason, 토큰 사용량)
# 셀 13의 response를 사용합니다
print('=== 응답 메타데이터 탐색 ===')
print()

print('[response_metadata]')
for key, value in response.response_metadata.items():
    print(f'  {key}: {value}')

print()
print('[usage_metadata]')
usage = response.usage_metadata
if usage:
    print(f'  입력 토큰:  {usage.get("input_tokens", "N/A")}')
    print(f'  출력 토큰:  {usage.get("output_tokens", "N/A")}')
    print(f'  총 토큰:    {usage.get("total_tokens", "N/A")}')
else:
    print('  (토큰 정보 없음)')

print()
print(f'finish_reason: {response.response_metadata.get("finish_reason", "N/A")}')
print('  - STOP: 정상 완료')
print('  - MAX_TOKENS: 길이 제한으로 중단됨')

=== 응답 메타데이터 탐색 ===

[response_metadata]
  finish_reason: STOP
  model_name: gemini-2.5-flash
  safety_ratings: []
  model_provider: google_genai

[usage_metadata]
  입력 토큰:  43
  출력 토큰:  1601
  총 토큰:    1644

finish_reason: STOP
  - STOP: 정상 완료
  - MAX_TOKENS: 길이 제한으로 중단됨


---

## 섹션 7: stream() 실시간 출력

### stream()이란?

`invoke()`와 `stream()`은 같은 결과를 반환하지만, **받는 방식**이 다릅니다.

**주문 배달 비유**
- `invoke()` = 피자 배달: 피자가 **완성되면** 한 번에 받음. 기다리는 동안 아무것도 안 보임
- `stream()` = 초밥 회전 레일: 만들어지는 대로 **조금씩** 나옴. 기다릴 필요 없이 바로바로 먹을 수 있음

### 언제 stream()을 쓰나요?

| 상황 | 추천 방법 |
|------|----------|
| 긴 글 생성 (보고서, 소설 등) | `stream()` - 기다리지 않고 읽을 수 있음 |
| 챗봇 UI (사용자가 실시간 확인) | `stream()` - 빠르게 응답받는 느낌 |
| 짧은 답변, 후처리 필요 | `invoke()` - 전체 결과를 한 번에 다루기 편함 |

```python
# invoke(): 완성될 때까지 기다렸다가 한 번에 출력
response = llm.invoke("...")
print(response.content)

# stream(): 생성되는 대로 조각(chunk)이 계속 들어옴
for chunk in llm.stream("..."):
    print(chunk.content, end="", flush=True)
```

In [13]:
# stream()으로 실시간 출력
# 텍스트가 조금씩 출력되는 것을 확인하세요
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content='당신은 창의적인 작가입니다. 생동감 있고 구체적으로 묘사하세요.'),
    HumanMessage(content='LangChain을 배우는 개발자의 하루를 3~4문장으로 짧게 묘사해주세요.'),
]

print('=== stream() 실시간 출력 ===')
print('(텍스트가 조금씩 나타납니다)')
print()

chunk_count = 0
full_text = ''

for chunk in llm.stream(messages):
    if chunk.content:
        print(chunk.content, end='', flush=True)
        full_text += chunk.content
        chunk_count += 1

print()
print()
print(f'[OK] 스트리밍 완료! 총 {chunk_count}개 청크로 나뉘어 수신했습니다.')
print(f'전체 텍스트 길이: {len(full_text)} 글자')

=== stream() 실시간 출력 ===
(텍스트가 조금씩 나타납니다)

아침 햇살 아래, 그는 IDE를 켜고 새로운 LangChain 에이전트의 뼈대를 세우기 시작합니다. 복잡한 체인과 프롬프트 템플릿 사이에서 씨름하며, 때로는 예상치 못한 LLM의 '환각'에 머리를 긁적이기도 하지만, VectorStore와 Retriever를 엮어 정보 검색의 정확도를 높이는 데 몰두합니다. 수많은 시도 끝에 마침내 원하는 답변을 매끄럽게 뱉어내는 파이프라인을 보며 작은 전기적 희열과 함께 안도의 한숨을 내쉽니다. 오늘 하루도 그는 인공지능이 가진 무한한 가능성의 조각을 하나 더 맞춰냈습니다.

[OK] 스트리밍 완료! 총 5개 청크로 나뉘어 수신했습니다.
전체 텍스트 길이: 287 글자


---

## 섹션 8: 마무리

### 배운 것 정리

모듈 01에서 배운 핵심 내용을 정리합니다.

#### ChatGoogleGenerativeAI 초기화
```python
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',  # 모델 선택
    temperature=0.7,           # 창의성 (0.0~2.0)
)
```

#### 두 가지 invoke() 방법 비교

| 구분 | 문자열 입력 | 메시지 리스트 입력 |
|------|-------------|--------------------|
| 코드 | `llm.invoke("질문")` | `llm.invoke([SystemMessage(...), HumanMessage(...)])` |
| 역할 설정 | 불가 | SystemMessage로 가능 |
| 사용 상황 | 간단한 테스트 | 실제 서비스, 역할 필요 시 |
| 반환 타입 | AIMessage | AIMessage |

#### invoke() vs stream() 비교

| 구분 | invoke() | stream() |
|------|----------|----------|
| 출력 시점 | 완성 후 한 번에 | 생성되는 대로 조금씩 |
| 반환 타입 | AIMessage | AIMessage 청크들 (iterator) |
| 사용 상황 | 짧은 응답, 후처리 필요 | 긴 글, 실시간 UI |

#### 핵심 코드 패턴

```python
# 공통: 결과 텍스트는 .content로 꺼냅니다
response = llm.invoke(...)  # AIMessage 객체
print(response.content)     # 텍스트만 꺼내기
```

## 다음 모듈 예고: 모듈 02 - 프롬프트 템플릿

모듈 01에서는 매번 직접 문자열을 써서 AI를 호출했습니다.
다음 모듈에서는 **재사용 가능한 프롬프트 템플릿**을 만드는 방법을 배웁니다.

### 모듈 02에서 배울 것

```python
from langchain_core.prompts import ChatPromptTemplate

# 변수를 포함한 재사용 가능한 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {language} 전문가입니다.'),
    ('human', '{topic}에 대해 설명해주세요.'),
])

# 변수를 채워서 사용
filled = prompt.invoke({'language': 'Python', 'topic': '클래스'})
response = llm.invoke(filled)
```

### 준비 체크리스트

다음 모듈로 넘어가기 전에 확인하세요:

- [ ] 셀 8: 문자열 invoke()에서 Gemini 응답이 출력되었나요?
- [ ] 셀 13: 메시지 리스트 invoke()가 성공했나요?
- [ ] 셀 14: SystemMessage를 바꿨을 때 응답이 달라졌나요?
- [ ] 셀 18: stream()으로 텍스트가 조금씩 출력되었나요?

모두 체크했다면 `02_prompts/모듈02_프롬프트_템플릿.ipynb`를 열어보세요!

---

수고하셨습니다!

> 질문이나 오류가 생기면 커리큘럼.md의 참고 자료를 확인하거나, [LangChain 공식 문서](https://python.langchain.com/)를 참고하세요.